<div style="font-family:'Segoe UI',Roboto,Helvetica,Arial,sans-serif;max-width:900px;margin:0 auto;border-radius:16px;overflow:hidden;box-shadow:0 4px 20px rgba(0,0,0,0.12);border:1px solid #e2e2e2;">

<div style="background:linear-gradient(135deg,#002855 0%,#004b8d 55%,#0077c8 100%);padding:28px 20px 22px 20px;text-align:center;">
<img src="./img/ITESOLogo.png" alt="ITESO" width="260" style="margin-bottom:10px;">
<div style="color:#ffffff;font-size:15px;font-weight:600;letter-spacing:0.5px;text-transform:uppercase;opacity:0.9;">
Departamento de Electrónica, Sistemas e Informática
</div>
</div>

<div style="background-color:#ffffff;padding:26px 30px 30px 30px;text-align:center;">

<div style="color:#002855;font-size:26px;font-weight:800;margin-bottom:6px;">
Big Data Analysis
</div>

<div style="display:inline-block;background-color:#eaf4fb;color:#0077c8;font-size:13px;font-weight:700;padding:4px 14px;border-radius:20px;letter-spacing:0.5px;margin-bottom:22px;">
Autumn 2026
</div>


<hr style="border:none;border-top:2px solid #f0f0f0;margin:0 0 22px 0;">

<div style="background-color:#f7fafd;border-left:5px solid #0077c8;border-radius:8px;padding:14px 18px;text-align:left;margin-bottom:18px;">
<div style="font-size:12px;color:#7a7a7a;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:4px;">
Session 09
</div>
<div style="font-size:19px;color:#002855;font-weight:700;">
Batch Processing
</div>
</div>

<div style="font-size:14px;color:#444;margin-top:20px;">
<span style="font-weight:700;color:#002855;">Profesor:</span> Pablo Camarillo Ramírez
</div>

</div>
</div>

In [2]:
#from pcamarillor.spark_utils import SparkUtils
import findspark
findspark.init()
from pyspark.sql import SparkSession

# Create Spark session
spark = SparkSession.builder \
            .appName("Batch processing") \
            .master("local[*]") \
            .config("spark.ui.port", "4040") \
            .getOrCreate()

sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 03:22:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Create the schema
columns_info = [
    ("date", "date"),
    ("hour", "string"),
    ("passenger_count", "double"),
    ("PU_Borough", "string"),
    ("DO_Borough", "string"),
    ("payment_type", "int"),
    ("trip_count", "int"),
    ("trip_distance_sum", "double"),
    ("duration_sum", "double"),
    ("fare_amount_sum", "double"),
    ("extra_sum", "double"),
    ("mta_tax_sum", "double"),
    ("tip_amount_sum", "double"),
    ("tolls_amount_sum", "double"),
    ("improvement_surcharge_sum", "double"),
    ("congestion_surcharge_sum", "double"),
    ("airport_fee_sum", "double"),
    ("total_amount_sum", "double"),
]
#aggregated_trips_schema = SparkUtils.generate_schema(columns_info)
df_nyc_taxi = (
    spark.read
    #.schema(aggregated_trips_schema)
    .option("inferSchema", "true") # <--- This option should be avoided in future pipelines
    .option("header", "true")
    .csv("/opt/spark/work-dir/data/nyc_taxi/archive/aggregated_nyc_yellow_taxi_2024.csv")
)
df_nyc_taxi.printSchema()
df_nyc_taxi.show(2)

root
 |-- date: date (nullable = true)
 |-- hour: timestamp (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- PU_Borough: string (nullable = true)
 |-- DO_Borough: string (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_count: integer (nullable = true)
 |-- trip_distance_sum: double (nullable = true)
 |-- duration_sum: double (nullable = true)
 |-- fare_amount_sum: double (nullable = true)
 |-- extra_sum: double (nullable = true)
 |-- mta_tax_sum: double (nullable = true)
 |-- tip_amount_sum: double (nullable = true)
 |-- tolls_amount_sum: double (nullable = true)
 |-- improvement_surcharge_sum: double (nullable = true)
 |-- congestion_surcharge_sum: double (nullable = true)
 |-- airport_fee_sum: double (nullable = true)
 |-- total_amount_sum: double (nullable = true)

+----------+-------------------+---------------+----------+----------+------------+----------+-----------------+------------------+---------------+---------+-----------+-------

## Data Cleaning

In [4]:
n_records = df_nyc_taxi.count()
n_records

921373

In [5]:
from pyspark.sql.functions import count, when, isnull

df_nyc_taxi.select([count(when(isnull(c), c)).alias(c) for c in df_nyc_taxi.columns]).show()


+----+----+---------------+----------+----------+------------+----------+-----------------+------------+---------------+---------+-----------+--------------+----------------+-------------------------+------------------------+---------------+----------------+
|date|hour|passenger_count|PU_Borough|DO_Borough|payment_type|trip_count|trip_distance_sum|duration_sum|fare_amount_sum|extra_sum|mta_tax_sum|tip_amount_sum|tolls_amount_sum|improvement_surcharge_sum|congestion_surcharge_sum|airport_fee_sum|total_amount_sum|
+----+----+---------------+----------+----------+------------+----------+-----------------+------------+---------------+---------+-----------+--------------+----------------+-------------------------+------------------------+---------------+----------------+
|   0|   0|              0|         0|         0|           0|         0|                0|           0|              0|        0|          0|             0|               0|                        0|                       

In [ ]:
df_nyc_taxi.count()

In [ ]:
clean = df_nyc_taxi.dropna()
clean.select([count(when(isnull(c), c)).alias(c) for c in clean.columns]).show()

In [ ]:
clean.count()

In [ ]:
clean_fillna = df_nyc_taxi.fillna({
    'passenger_count': 0
})
clean_fillna.count()

In [ ]:
clean_fillna.select([count(when(isnull(c), c)).alias(c) for c in clean_fillna.columns]).show()

# Basic Transformations

In [7]:
from pyspark.sql.functions import col, lit, when

## Select

In [8]:
# Pick a subset of columns to work with
trip_summary_df = df_nyc_taxi.select(
    "date", "hour", "PU_Borough", "DO_Borough", "trip_count", "total_amount_sum"
)
trip_summary_df.show(5)

# select() with col() to reference columns explicitly (useful when combining
# with expressions or disambiguating columns after a join)
fare_df = df_nyc_taxi.select(
    col("date"),
    col("PU_Borough"),
    col("fare_amount_sum"),
    col("tip_amount_sum")
)
fare_df.show(5)

+----------+-------------------+----------+----------+----------+-----------------+
|      date|               hour|PU_Borough|DO_Borough|trip_count| total_amount_sum|
+----------+-------------------+----------+----------+----------+-----------------+
|2024-01-01|2026-09-22 00:00:00|     Bronx|     Bronx|         2|             18.0|
|2024-01-01|2026-09-22 00:00:00|     Bronx| Manhattan|         1|             34.9|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|  Brooklyn|         6|87.35000000000001|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|  Brooklyn|         4|            80.85|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn| Manhattan|        10|           436.58|
+----------+-------------------+----------+----------+----------+-----------------+
only showing top 5 rows
+----------+----------+---------------+--------------+
|      date|PU_Borough|fare_amount_sum|tip_amount_sum|
+----------+----------+---------------+--------------+
|2024-01-01|     Bronx|           13.0|           0.0|


## withColumn + lit()

In [9]:
# --- withColumn() + lit() ---
# Add a constant/literal column, e.g. tagging the source of this batch
tagged_df = df_nyc_taxi.withColumn("data_source", lit("nyc_tlc_2024"))
tagged_df.select("date", "PU_Borough", "data_source").show(5)

# Derive a new numeric column: average fare per trip
avg_fare_df = df_nyc_taxi.withColumn(
    "avg_fare_per_trip",
    col("fare_amount_sum") / col("trip_count")
)
avg_fare_df.select("date", "hour", "trip_count", "fare_amount_sum", "avg_fare_per_trip").show(5)


+----------+----------+------------+
|      date|PU_Borough| data_source|
+----------+----------+------------+
|2024-01-01|     Bronx|nyc_tlc_2024|
|2024-01-01|     Bronx|nyc_tlc_2024|
|2024-01-01|  Brooklyn|nyc_tlc_2024|
|2024-01-01|  Brooklyn|nyc_tlc_2024|
|2024-01-01|  Brooklyn|nyc_tlc_2024|
+----------+----------+------------+
only showing top 5 rows
+----------+-------------------+----------+---------------+------------------+
|      date|               hour|trip_count|fare_amount_sum| avg_fare_per_trip|
+----------+-------------------+----------+---------------+------------------+
|2024-01-01|2026-09-22 00:00:00|         2|           13.0|               6.5|
|2024-01-01|2026-09-22 00:00:00|         1|           32.4|              32.4|
|2024-01-01|2026-09-22 00:00:00|         6|           60.7|10.116666666666667|
|2024-01-01|2026-09-22 00:00:00|         4|           70.6|             17.65|
|2024-01-01|2026-09-22 00:00:00|        10|          303.0|              30.3|
+----------

## When

In [10]:
# --- when() ---
# Recode payment_type (1 = credit card, 2 = cash, per TLC docs) into a readable label
payment_label_df = df_nyc_taxi.withColumn(
    "payment_label",
    when(col("payment_type") == 1, lit("Credit Card"))
    .when(col("payment_type") == 2, lit("Cash"))
    .otherwise(lit("Other"))
)
payment_label_df.select("payment_type", "payment_label").distinct().show()

# Flag trips that crossed a bridge/tunnel or airport, based on summed surcharges
flagged_df = df_nyc_taxi.withColumn(
    "trip_type",
    when(col("airport_fee_sum") > 0, lit("Airport"))
    .when(col("tolls_amount_sum") > 0, lit("Toll Road"))
    .otherwise(lit("Standard"))
)
flagged_df.select("date", "hour", "PU_Borough", "DO_Borough", "trip_type").show(10)

+------------+-------------+
|payment_type|payment_label|
+------------+-------------+
|           0|        Other|
|           1|  Credit Card|
|           4|        Other|
|           2|         Cash|
|           3|        Other|
+------------+-------------+

+----------+-------------------+----------+----------+---------+
|      date|               hour|PU_Borough|DO_Borough|trip_type|
+----------+-------------------+----------+----------+---------+
|2024-01-01|2026-09-22 00:00:00|     Bronx|     Bronx| Standard|
|2024-01-01|2026-09-22 00:00:00|     Bronx| Manhattan| Standard|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|  Brooklyn| Standard|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|  Brooklyn|  Airport|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn| Manhattan|Toll Road|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn| Manhattan| Standard|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|    Queens| Standard|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|    Queens| Standard|
|2024-01-01|2026-09-22 

## Consolidated example

In [11]:
enriched_df = (
    df_nyc_taxi
    .withColumn(
        "borough_match",
        when(col("PU_Borough") == col("DO_Borough"), lit("Same Borough"))
        .otherwise(lit("Cross Borough"))
    )
    .select(
        col("date"),
        col("hour"),
        col("PU_Borough"),
        col("DO_Borough"),
        col("borough_match"),
        col("trip_count"),
        col("total_amount_sum")
    )
)
enriched_df.show(10)

+----------+-------------------+----------+----------+-------------+----------+------------------+
|      date|               hour|PU_Borough|DO_Borough|borough_match|trip_count|  total_amount_sum|
+----------+-------------------+----------+----------+-------------+----------+------------------+
|2024-01-01|2026-09-22 00:00:00|     Bronx|     Bronx| Same Borough|         2|              18.0|
|2024-01-01|2026-09-22 00:00:00|     Bronx| Manhattan|Cross Borough|         1|              34.9|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|  Brooklyn| Same Borough|         6| 87.35000000000001|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|  Brooklyn| Same Borough|         4|             80.85|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn| Manhattan|Cross Borough|        10|            436.58|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn| Manhattan|Cross Borough|         1|              12.9|
|2024-01-01|2026-09-22 00:00:00|  Brooklyn|    Queens|Cross Borough|         2|             61.96|
|2024-01-0

26/09/24 00:34:15 WARN NettyRpcEnv: Ignored failure: java.util.concurrent.TimeoutException: Cannot receive any reply from 79b64abf62f0:33057 in 10000 milliseconds
26/09/24 00:35:42 WARN NettyRpcEnv: Ignored message: HeartbeatResponse(false)
26/09/24 00:35:56 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.rpc.RpcTimeoutException: Future timed out after [10000 milliseconds]. This timeout is controlled by spark.executor.heartbeatInterval
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:62)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIfTimeout$1.applyOrElse(RpcTimeout.scala:58)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:76)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef

In [38]:
sc.stop()

26/09/22 03:22:30 WARN JavaUtils: Attempt to delete using native Unix OS command failed for path = /tmp/spark-28c0131a-5383-4f69-8728-004b20a52fff/pyspark-37b4e058-f59c-41bd-94ce-43b7800304de. Falling back to Java IO way
java.io.IOException: Failed to delete: /tmp/spark-28c0131a-5383-4f69-8728-004b20a52fff/pyspark-37b4e058-f59c-41bd-94ce-43b7800304de
	at org.apache.spark.network.util.JavaUtils.deleteRecursivelyUsingUnixNative(JavaUtils.java:352)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:269)
	at org.apache.spark.network.util.JavaUtils.deleteRecursively(JavaUtils.java:248)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively(SparkFileUtils.scala:158)
	at org.apache.spark.util.SparkFileUtils.deleteRecursively$(SparkFileUtils.scala:157)
	at org.apache.spark.util.Utils$.deleteRecursively(Utils.scala:1062)
	at org.apache.spark.util.ShutdownHookManager$.$anonfun$new$4(ShutdownHookManager.scala:70)
	at org.apache.spark.util.ShutdownHookManager$.$anonfu